# Repair Agent
The task of the repair agent is to fix the syntax and some semantic errors. It receives a list of errors or suggestions and tries to improve the result. One technique that is used a lot in SWE Agents is the simple diff-format.

```
5c5
< alter Text
---
> neuer Text
```

In [1]:
from pathlib import Path
import sys
import logging

logging.basicConfig(level=logging.INFO)

ROOT = Path.cwd().parent.parent
logging.info(f"Adding {ROOT} to Python path for imports in kernel.")
sys.path.insert(0, str(ROOT))  # ROOT, nicht SRC!

# Load config with changed env path
from src.config import Config
config = Config.get_instance(env_path=ROOT / ".env")


INFO:root:Adding /Users/lukas/Masterarbeit/agents/bxAgent to Python path for imports in kernel.


After the module initialization, the agent should be build and the setup has to be prepared. For that we need a filesystem backend so that the agent can write arbitrary files.

In [2]:
from deepagents.backends import LocalShellBackend

backend = LocalShellBackend(
    root_dir=config.WORKSPACE.PATH / "transformation",
    virtual_mode=True
)

In [3]:
from src.agents.repair import build_repair_agent

agent = build_repair_agent(backend=backend)

Let's invoke the agent by providing a prompt with the containing paths. Some does not exist.

**Changes**:
- 2026-05-14: Instead of providing the files within the prompt, use a Custom Agent State and invoke the agent with the files to make it more stable.

In [4]:
from langchain.messages import HumanMessage

prompt = """
Here are some files that contain errors: Please fix them!
/transformation/FamiliesToPersonsTransformation.java containing error:
FamiliesToPersonsTransformation.java:5: Fehler: Package persons.Factories ist nicht vorhanden
import persons.Factories.PersonFactory;
                        ^

/transformation/PersonsToFamiliesTransformation.java containing error:
FamiliesToPersonsTransformation.java:6: Fehler: Package persons ist nicht vorhanden
import persons.Person;
              ^
              
/transformation/PersonsTest.java containing error: FileNotFoundError: No such file or directory: 'transformation/PersonsTest.java'
"""

agent.invoke(
    input={"messages": [HumanMessage(content=prompt)]},
    config={"configurable": {"thread_id": "some_id"}}
)

INFO:httpx:HTTP Request: POST https://chat-1.ki-awz.iisys.de/api/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://chat-1.ki-awz.iisys.de/api/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://chat-1.ki-awz.iisys.de/api/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://chat-1.ki-awz.iisys.de/api/chat/completions "HTTP/1.1 200 OK"


{'messages': [HumanMessage(content="\nHere are some files that contain errors: Please fix them!\n/transformation/FamiliesToPersonsTransformation.java containing error:\nFamiliesToPersonsTransformation.java:5: Fehler: Package persons.Factories ist nicht vorhanden\nimport persons.Factories.PersonFactory;\n                        ^\n\n/transformation/PersonsToFamiliesTransformation.java containing error:\nFamiliesToPersonsTransformation.java:6: Fehler: Package persons ist nicht vorhanden\nimport persons.Person;\n              ^\n\n/transformation/PersonsTest.java containing error: FileNotFoundError: No such file or directory: 'transformation/PersonsTest.java'\n", additional_kwargs={}, response_metadata={}, id='9d182acc-5f45-4a18-98db-01a7ec726fd8'),
  AIMessage(content='<think>The user wants me to fix errors in three Java files. Let me analyze each error:\n\n1. **FamiliesToPersonsTransformation.java**: Error says "Package persons.Factories is not present" - the import is trying to use `pe